### D405랑 D457비교

In [6]:
import pyrealsense2 as rs

ctx = rs.context()
devices = ctx.query_devices()
print("num devices:", len(devices))

for i, dev in enumerate(devices):
    name = dev.get_info(rs.camera_info.name)
    sn   = dev.get_info(rs.camera_info.serial_number)
    pid  = dev.get_info(rs.camera_info.product_id)
    usb  = dev.get_info(rs.camera_info.usb_type_descriptor)
    print(f"[{i}] name={name}, sn={sn}, pid={pid}, usb={usb}")


num devices: 1
[0] name=Intel RealSense D405, sn=335122272982, pid=0B5B, usb=3.2


In [ ]:
# box_live_overlay_standalone.py
from __future__ import annotations
from typing import Optional, Dict
import os
import time
import traceback
from datetime import datetime

import numpy as np
import cv2
import pyrealsense2 as rs
from ultralytics import YOLO


# ============================================================
# ✅ Embedded vision_config (import 없이 이 파일 안에 박아둠)
# ============================================================
class CFG:
    # -------------------------
    # Model
    # -------------------------
    MODEL_PATH = "/home/dw/ws_job_msislab/amr_project/src/job_pc/runs/obb/smoke_test_v2/weights/best.pt"

    CONF_THRES = 0.85
    IOU_THRES  = 0.85
    IMGSZ      = 640

    # -------------------------
    # Camera stream (D405)
    # -------------------------
    WIDTH  = 640
    HEIGHT = 480
    FPS    = 30

    # -------------------------
    # Depth ROI (meters)
    # -------------------------
    ROI_MARGIN_PX  = 6.0
    MIN_ROI_PIXELS = 120
    MAD_THRES_M    = 0.020
    DEPTH_MIN_M = 0.15
    DEPTH_MAX_M = 3.00

    # -------------------------
    # Sanity / jump filters
    # -------------------------
    Z_RANGE_MM = (150.0, 1200.0)
    JUMP_XY_MM   = 35.0
    JUMP_Z_MM    = 60.0
    JUMP_ANG_DEG = 10.0
    MAX_CONSEC_SKIPS_RESET = 15  # 여기선 stale 표시용

    # -------------------------
    # Preview / overlay
    # -------------------------
    SHOW_PREVIEW = True
    PREVIEW_WIN_NAME = "Box Live Overlay"

    SHOW_OVERLAY = True
    OVERLAY_FONT_SCALE = 0.6
    OVERLAY_THICKNESS  = 2

    PRINT_SELECTED_EACH_ACCEPT = False

    # -------------------------
    # Capture output
    # -------------------------
    OUT_DIR = "./captures"


# =========================================================
# Helpers (measure_box_2.py와 동일 계열)
# =========================================================
def poly_shrink_towards_center(poly4x2: np.ndarray, margin_px: float):
    p = poly4x2.astype(np.float32)
    c = p.mean(axis=0, keepdims=True)
    v = p - c
    norm = np.linalg.norm(v, axis=1, keepdims=True) + 1e-6
    return p - (v / norm) * margin_px


def depth_roi_stats(depth_u16: np.ndarray, depth_scale: float, poly4x2: np.ndarray):
    h, w = depth_u16.shape[:2]
    poly = np.round(poly4x2).astype(np.int32)

    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [poly.reshape(-1, 1, 2)], 255)

    d = depth_u16[mask == 255].astype(np.float32) * depth_scale
    d = d[(d > 0) & (d >= float(CFG.DEPTH_MIN_M)) & (d <= float(CFG.DEPTH_MAX_M))]

    if d.size == 0:
        return 0.0, 0.0, 0

    med = float(np.median(d))
    mad = float(np.median(np.abs(d - med)))
    return med, mad, int(d.size)


def XY_from_pixel_and_Z(cx: float, cy: float, intr, Z_m: float):
    X = (cx - intr.ppx) / intr.fx * Z_m
    Y = (cy - intr.ppy) / intr.fy * Z_m
    return float(X), float(Y)


def obb_angle_deg_upright0_rightplus(poly4x2: np.ndarray) -> float:
    p = poly4x2.astype(np.float32)
    c = p.mean(axis=0, keepdims=True)
    q = p - c
    cov = np.cov(q.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    v = eigvecs[:, np.argmax(eigvals)].astype(np.float32)
    vx, vy = float(v[0]), float(v[1])
    if vy < 0:
        vx, vy = -vx, -vy
    return -float(np.degrees(np.arctan2(vx, vy)))


def is_jump(prev: Optional[Dict[str, float]], cur: Dict[str, float]) -> bool:
    if prev is None:
        return False
    if abs(cur["move_x_mm"] - prev["move_x_mm"]) > float(CFG.JUMP_XY_MM):
        return True
    if abs(cur["move_y_mm"] - prev["move_y_mm"]) > float(CFG.JUMP_XY_MM):
        return True
    if abs(cur["move_z_mm"] - prev["move_z_mm"]) > float(CFG.JUMP_Z_MM):
        return True
    if abs(cur.get("angle_deg", 0.0) - prev.get("angle_deg", 0.0)) > float(CFG.JUMP_ANG_DEG):
        return True
    return False


def ts_str():
    return datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:-3]


def ensure_outdir():
    outdir = str(getattr(CFG, "OUT_DIR", "./captures"))
    os.makedirs(outdir, exist_ok=True)
    return outdir


def try_open_writer(base_path_no_ext: str, w: int, h: int, fps: float):
    """
    mp4 우선, 안 되면 avi로 fallback.
    """
    # 1) mp4v
    mp4_path = base_path_no_ext + ".mp4"
    fourcc_mp4 = cv2.VideoWriter_fourcc(*"mp4v")
    wr = cv2.VideoWriter(mp4_path, fourcc_mp4, float(fps), (int(w), int(h)))
    if wr.isOpened():
        return wr, mp4_path

    # 2) XVID avi
    avi_path = base_path_no_ext + ".avi"
    fourcc_avi = cv2.VideoWriter_fourcc(*"XVID")
    wr2 = cv2.VideoWriter(avi_path, fourcc_avi, float(fps), (int(w), int(h)))
    if wr2.isOpened():
        return wr2, avi_path

    # 둘 다 실패
    try:
        wr.release()
    except Exception:
        pass
    try:
        wr2.release()
    except Exception:
        pass
    return None, ""


# =========================================================
# Main: XYZ/Angle 계속 띄우기 + Screenshot/Record
# =========================================================
def main():
    if os.environ.get("DISPLAY", "") == "":
        print("[WARN] DISPLAY가 비어있음. (SSH라면 X11 forwarding 또는 VNC 필요)")

    outdir = ensure_outdir()

    model_path = CFG.MODEL_PATH
    if not model_path:
        raise RuntimeError("CFG.MODEL_PATH가 비어있음")

    print(f"[Live] YOLO OBB 로딩: {model_path}")
    model = YOLO(model_path, task="obb")

    w, h, fps = int(CFG.WIDTH), int(CFG.HEIGHT), int(CFG.FPS)

    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_stream(rs.stream.color, w, h, rs.format.bgr8, fps)
    config.enable_stream(rs.stream.depth, w, h, rs.format.z16, fps)

    print(f"[Live] RealSense start: {w}x{h}@{fps}")
    profile = pipeline.start(config)

    depth_scale = float(profile.get_device().first_depth_sensor().get_depth_scale())
    align = rs.align(rs.stream.color)

    spatial = rs.spatial_filter()
    temporal = rs.temporal_filter()

    win = str(CFG.PREVIEW_WIN_NAME)
    cv2.namedWindow(win, cv2.WINDOW_NORMAL)

    last_valid: Optional[Dict[str, float]] = None
    last_valid_t = 0.0
    consec_miss = 0

    t_prev = time.time()
    fps_smooth = 0.0

    # recording
    is_recording = False
    writer: Optional[cv2.VideoWriter] = None
    record_path = ""

    print("[Keys] ESC: quit | s: screenshot | r: record toggle")

    try:
        while True:
            frames = pipeline.wait_for_frames(1000)
            aligned = align.process(frames)

            color_frame = aligned.get_color_frame()
            depth_frame = aligned.get_depth_frame()
            if not color_frame or not depth_frame:
                continue

            img = np.asanyarray(color_frame.get_data())

            d_frame = spatial.process(depth_frame)
            d_frame = temporal.process(d_frame)
            d_u16 = np.asanyarray(d_frame.get_data())

            intr = color_frame.profile.as_video_stream_profile().get_intrinsics()

            H, W = img.shape[:2]
            IMG_CENTER_X = W / 2.0
            IMG_CENTER_Y = H / 2.0

            vis = img.copy()

            results = model.predict(
                img,
                imgsz=int(CFG.IMGSZ),
                conf=float(CFG.CONF_THRES),
                iou=float(CFG.IOU_THRES),
                verbose=False,
            )
            r = results[0]

            best_sample: Optional[Dict[str, float]] = None
            best_poly = None
            best_cxcy = None

            if hasattr(r, "obb") and r.obb is not None:
                candidates = []
                for i, c in enumerate(r.obb.conf):
                    if float(c) < float(CFG.CONF_THRES):
                        continue

                    poly = r.obb.xyxyxyxy[i].cpu().numpy().reshape(4, 2)

                    poly_s = poly_shrink_towards_center(poly, float(CFG.ROI_MARGIN_PX))
                    poly_s[:, 0] = np.clip(poly_s[:, 0], 0, W - 1)
                    poly_s[:, 1] = np.clip(poly_s[:, 1], 0, H - 1)

                    z_m, mad, count = depth_roi_stats(d_u16, depth_scale, poly_s)

                    if z_m > 0 and count > int(CFG.MIN_ROI_PIXELS) and (mad <= float(CFG.MAD_THRES_M)):
                        cx = float(np.mean(poly[:, 0]))
                        cy = float(np.mean(poly[:, 1]))
                        dist_center = float(np.hypot(cx - IMG_CENTER_X, cy - IMG_CENTER_Y))
                        angle = obb_angle_deg_upright0_rightplus(poly)

                        candidates.append(
                            {"z_m": z_m, "poly": poly, "cx": cx, "cy": cy,
                             "dist_center": dist_center, "angle": angle}
                        )

                if candidates:
                    candidates.sort(key=lambda x: x["dist_center"])
                    best = candidates[0]

                    tx, ty = XY_from_pixel_and_Z(best["cx"], best["cy"], intr, best["z_m"])
                    raw = {
                        "move_x_mm": float(tx * 1000.0),
                        "move_y_mm": float(ty * 1000.0),
                        "move_z_mm": float(best["z_m"] * 1000.0),
                        "angle_deg": float(best["angle"]),
                    }

                    zlo, zhi = float(CFG.Z_RANGE_MM[0]), float(CFG.Z_RANGE_MM[1])
                    if zlo <= raw["move_z_mm"] <= zhi and (not is_jump(last_valid, raw)):
                        best_sample = raw
                        best_poly = best["poly"]
                        best_cxcy = (best["cx"], best["cy"])

                        last_valid = raw
                        last_valid_t = time.time()
                        consec_miss = 0

                        if bool(CFG.PRINT_SELECTED_EACH_ACCEPT):
                            print(f"[ACCEPT] {raw}")

            if best_sample is None:
                consec_miss += 1

            # FPS
            t_now = time.time()
            dt = max(1e-6, t_now - t_prev)
            t_prev = t_now
            fps_inst = 1.0 / dt
            fps_smooth = (0.9 * fps_smooth) + (0.1 * fps_inst) if fps_smooth > 0 else fps_inst

            # Overlay text
            if bool(CFG.SHOW_OVERLAY):
                if last_valid is not None:
                    age = t_now - last_valid_t
                    stale = (consec_miss >= int(CFG.MAX_CONSEC_SKIPS_RESET))

                    txt1 = f"XYZ(mm): {last_valid['move_x_mm']:.0f}, {last_valid['move_y_mm']:.0f}, {last_valid['move_z_mm']:.0f}"
                    txt2 = f"angle(deg): {last_valid['angle_deg']:.2f} | FPS:{fps_smooth:.1f} | age:{age:.2f}s"
                    if stale:
                        txt2 += " | STALE"

                    cv2.putText(vis, txt1, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                                float(CFG.OVERLAY_FONT_SCALE), (0, 255, 0), int(CFG.OVERLAY_THICKNESS))
                    cv2.putText(vis, txt2, (10, 60), cv2.FONT_HERSHEY_SIMPLEX,
                                float(CFG.OVERLAY_FONT_SCALE), (0, 255, 0), int(CFG.OVERLAY_THICKNESS))
                else:
                    cv2.putText(vis, "NO VALID SAMPLE", (10, 30),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

            # OBB + center point (✅ center를 빨간색으로)
            if best_sample is not None and best_poly is not None and best_cxcy is not None:
                cv2.polylines(vis, [np.int32(best_poly)], True, (0, 255, 0), 2)
                cx, cy = best_cxcy
                cv2.circle(vis, (int(cx), int(cy)), 6, (0, 0, 255), -1)  # ✅ RED

            # Recording indicator
            if is_recording:
                cv2.putText(vis, "REC", (W - 70, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
                cv2.circle(vis, (W - 25, 22), 6, (0, 0, 255), -1)

            # Write video frame (with overlay)
            if is_recording and writer is not None:
                writer.write(vis)

            cv2.imshow(win, vis)
            key = cv2.waitKey(1) & 0xFF

            if key == 27:  # ESC
                break

            elif key in (ord('s'), ord('S')):
                path = os.path.join(outdir, f"shot_{ts_str()}.png")
                ok = cv2.imwrite(path, vis)
                print(f"[Shot] {'OK' if ok else 'FAIL'}: {path}")

            elif key in (ord('r'), ord('R')):
                # toggle recording
                if not is_recording:
                    base = os.path.join(outdir, f"rec_{ts_str()}")
                    writer, record_path = try_open_writer(base, W, H, float(fps))
                    if writer is None:
                        print("[REC] VideoWriter open FAIL (codec 문제 가능).")
                        is_recording = False
                        record_path = ""
                    else:
                        is_recording = True
                        print(f"[REC] START: {record_path}")
                else:
                    is_recording = False
                    if writer is not None:
                        try:
                            writer.release()
                        except Exception:
                            pass
                    writer = None
                    print(f"[REC] STOP: {record_path}")
                    record_path = ""

    except Exception:
        print("[Live] exception:")
        print(traceback.format_exc())

    finally:
        # stop record if needed
        try:
            if writer is not None:
                writer.release()
        except Exception:
            pass

        try:
            pipeline.stop()
        except Exception:
            pass
        try:
            cv2.destroyAllWindows()
        except Exception:
            pass
        print("[Live] 종료")


if __name__ == "__main__":
    main()


[Live] YOLO OBB 로딩: /home/dw/ws_job_msislab/amr_project/src/job_pc/runs/obb/smoke_test_v2/weights/best.pt
[Live] RealSense start: 640x480@30
[Keys] ESC: quit | s: screenshot | r: record toggle
